In [ ]:
#Solo descomenta esta celda si quieres tu iniciar sesión personal en Google Colab

#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
#!pip install osmnx rasterio mapclassify contextily

# Uso de rasterio y reclasificación de datos

## Primer ejemplo

In [ ]:
import os
import zipfile
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
from shapely.geometry import box
import warnings
warnings.filterwarnings("ignore")

En este ejemplo, utilizaremos una capa de uso de suelo y la reclasificaremos para crear una nueva capa que represente diferentes categorías de uso de suelo.

![](../source/images/usosuelo-gdl.png)

Debemos asegurarnos de que el CRS de la capa esté en un sistema que sea compatible para interpretar ``_metros_`` como unidades, porque vamos a realizar operaciones para calcular áreas:

Generamos una nueva columna para guardar los resultados de la reclasificación:

## Segundo ejemplo

In [ ]:
## Este ejemplo solo es para demostrar cómo se puede descargar información a través de la conexión a Google Earth Engine desde Python.
## No podremos hacerlo en Google Colab porque no tenemos acceso a la API de Google Earth Engine desde este entorno.

In [ ]:
# Anteriormente, ya solicité el acceso a la API de Google Earth Engine y lo tengo activo.
# Para usar la API de Google Earth Engine, primero debemos instalar la biblioteca y autenticar nuestra cuenta.
import ee

In [ ]:
ee.Initialize(project='ee-paty-info-geo-sentinel')

### Caso: Área de Jalisco

In [ ]:
# 1. Definimos el área de interés (AOI) como un rectángulo que cubre una región específica.

# En este caso, tomaremos un área en Jalisco, México.
aoi = ee.Geometry.Rectangle([-103.6, 20.5, -103.2, 20.8]) 

In [ ]:
## Código primero

In [ ]:
# 2. Función para enmascarar nubes usando la banda QA60
def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).copyProperties(image, ["system:time_start"])

# 3. Filtrar colección Sentinel-2 y aplicar máscara de nubes
s2_clean = ee.ImageCollection("COPERNICUS/S2_SR") \
    .filterBounds(aoi) \
    .filterDate('2023-01-01', '2023-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(maskS2clouds)

# 4. Obtener la primera imagen limpia
image = s2_clean.first().select(['B4', 'B3', 'B2'])  # RGB

# 5. Exportar a mii Google Drive
task = ee.batch.Export.image.toDrive(
    image=image.clip(aoi),
    description='sentinel_rgb_sin_nubes_jalisco',
    folder='EarthEngine',
    fileNamePrefix='sentinel_jalisco_rgb_clean',
    region=aoi,
    scale=10,
    fileFormat='GeoTIFF'
)

task.start()

print("Tarea de exportación iniciada.")

In [ ]:
task.status()

### Caso: SRTM

#### Obtener imagen del satélite _Shuttle Radar Topography Mission (SRTM)_

``USGS/SRTMGL1_003`` es un conjunto de datos que contiene información de **elevación global**, con una resolución aproximada de 30 metros.

Introducimos también la biblioteca de `osmnx` para descargar datos sobre: **carreteras, caminos y parques**.

In [ ]:
# Código segundo

## Rasterio

In [ ]:
import rasterio

#### _Shuttle Radar Topography Mission (SRTM)_

[SRTM](https://www.usgs.gov/centers/eros/science/usgs-eros-archive-digital-elevation-shuttle-radar-topography-mission-srtm-1)

* Cada píxel de la imagen representa una altura en metros sobre el nivel del mar.

* Es un raster georreferenciado, lo que significa que cada píxel tiene una ubicación exacta en la Tierra.

* Resolución: 30 metros (cada píxel representa un cuadrado de 30x30 metros en el terreno).

Sombras en escala de grises: representan variación de altitud

* Negro / oscuro: zonas más bajas

* Blanco / claro: zonas elevadas (cerros, sierras)

![](../source/images/elev.png)

Vamos a "recortar" el área de interés entre `grid` y los objetos vectoriales `road` y `parks`

In [ ]:
# Código tercero

### Clasificación de mapas temáticos

In [ ]:
import mapclassify

Ahora, construyamos una función que nos permita reclasificar las tres características de interés:

In [ ]:
# Código cuarto

In [ ]:
import contextily as cx